### Connect to BigQuery 

In [ ]:
import pandas as pd


processed_path = "../data/processed"

# Load all 9 cleaned CSVs back into dataframes
companies = pd.read_csv(f"{processed_path}/companies.csv")
corporate_policies = pd.read_csv(f"{processed_path}/corporate_policies.csv")
covered_employees = pd.read_csv(f"{processed_path}/covered_employees.csv")
claims = pd.read_csv(f"{processed_path}/claims.csv")
policy_table = pd.read_csv(f"{processed_path}/policy_table.csv")
premium_payments = pd.read_csv(f"{processed_path}/premium_payments.csv")
branches = pd.read_csv(f"{processed_path}/branches.csv")
claim_workflow = pd.read_csv(f"{processed_path}/claim_workflow.csv")
beneficiaries = pd.read_csv(f"{processed_path}/beneficiaries.csv")

print("All 9 tables loaded.")

All 9 tables loaded.


In [2]:
from google.cloud import bigquery
import os

# Point to our credentials file
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../credentials/bigquery-key.json"

# Connect to BigQuery
client = bigquery.Client()

print("Connected to project:", client.project)

Connected to project: akwaaba-insurance-500201


### Create a dataset in BigQuery

In [3]:
dataset_id = f"{client.project}.akwaaba_insurance"

dataset = bigquery.Dataset(dataset_id)
dataset.location = "US" 

dataset = client.create_dataset(dataset, exists_ok=True)
print(f"Dataset ready: {dataset_id}")

Dataset ready: akwaaba-insurance-500201.akwaaba_insurance


### Load the 9 tables into BigQuery

In [4]:
tables_to_load = {
    'companies': companies,
    'corporate_policies': corporate_policies,
    'covered_employees': covered_employees,
    'claims': claims,
    'policy_table': policy_table,
    'premium_payments': premium_payments,
    'branches': branches,
    'claim_workflow': claim_workflow,
    'beneficiaries': beneficiaries
}

for name, df in tables_to_load.items():
    table_id = f"{dataset_id}.{name}"
    job = client.load_table_from_dataframe(df, table_id)
    job.result()  # waits for the load to finish
    print(f"Loaded {name} -> {table_id}")

Loaded companies -> akwaaba-insurance-500201.akwaaba_insurance.companies
Loaded corporate_policies -> akwaaba-insurance-500201.akwaaba_insurance.corporate_policies
Loaded covered_employees -> akwaaba-insurance-500201.akwaaba_insurance.covered_employees
Loaded claims -> akwaaba-insurance-500201.akwaaba_insurance.claims
Loaded policy_table -> akwaaba-insurance-500201.akwaaba_insurance.policy_table
Loaded premium_payments -> akwaaba-insurance-500201.akwaaba_insurance.premium_payments
Loaded branches -> akwaaba-insurance-500201.akwaaba_insurance.branches
Loaded claim_workflow -> akwaaba-insurance-500201.akwaaba_insurance.claim_workflow
Loaded beneficiaries -> akwaaba-insurance-500201.akwaaba_insurance.beneficiaries


In [5]:
## Verify the loads

for name in tables_to_load.keys():
    table_id = f"{dataset_id}.{name}"
    table = client.get_table(table_id)
    print(f"{name}: {table.num_rows} rows, {len(table.schema)} columns")

companies: 7 rows, 3 columns
corporate_policies: 70 rows, 8 columns
covered_employees: 1974 rows, 13 columns
claims: 2500 rows, 6 columns
policy_table: 70 rows, 5 columns
premium_payments: 420 rows, 6 columns
branches: 70 rows, 3 columns
claim_workflow: 165 rows, 6 columns
beneficiaries: 1974 rows, 5 columns
